In [3]:
# ex3 problem 5 part a
# coding the dynamics function for example 3.5

# given code

from scipy.stats import poisson
import numpy as np
from enum import IntEnum
from typing import Tuple


class Action(IntEnum):
    """Action"""

    LEFT = 0
    DOWN = 1
    RIGHT = 2
    UP = 3


def actions_to_dxdy(action: Action) -> Tuple[int, int]:
    """
    Helper function to map action to changes in x and y coordinates
    Args:
        action (Action): taken action
    Returns:
        dxdy (Tuple[int, int]): Change in x and y coordinates
    """
    mapping = {
        Action.LEFT: (-1, 0),
        Action.DOWN: (0, -1),
        Action.RIGHT: (1, 0),
        Action.UP: (0, 1),
    }
    return mapping[action]


class Gridworld5x5:
    """5x5 Gridworld"""

    def __init__(self) -> None:
        """
        State: (x, y) coordinates

        Actions: See class(Action).
        """
        self.rows = 5
        self.cols = 5
        self.state_space = [
            (x, y) for x in range(0, self.rows) for y in range(0, self.cols)
        ]
        self.action_space = len(Action)
            
        # TODO set the locations of A and B, the next locations, and their rewards
        self.A = (1, 4)
        self.A_prime = (1, 0)
        self.A_reward = 10
        self.B = (3, 4)
        self.B_prime = (4, 2)
        self.B_reward = 5

    def transitions(
        self, state: Tuple, action: Action
    ) -> Tuple[Tuple[int, int], float]:
        """Get transitions from given (state, action) pair.

        Note that this is the 4-argument transition version p(s',r|s,a).
        This particular environment has deterministic transitions

        Args:
            state (Tuple): state
            action (Action): action

        Returns:
            next_state: Tuple[int, int]
            reward: float
        """
        state = self.state_space[state]
        next_state = None
        reward = None

        #print(f"dxdy: {actions_to_dxdy(action)}")
        next_state = tuple(map(sum, zip(state, actions_to_dxdy(action))))
        
        # TODO Check if current state is A and B and return the next state and corresponding reward
        if state == self.A:
            next_state = self.A_prime
            reward = 10
        elif state == self.B:
            next_state = self.B_prime
            reward = 5
            
        # Else, check if the next step is within boundaries and return next state and reward
        elif next_state[0] < 0 or next_state[0] > 4:
            next_state = state
            reward = -1
        elif next_state[1] < 0 or next_state[1] > 4:
            next_state = state
            reward = -1
        
        if reward == None:
            reward = 0
        
        return next_state, reward

    def expected_return(
        self, V, state: Tuple[int, int], action: Action, gamma: float
    ) -> float:
        """Compute the expected_return for all transitions from the (s,a) pair, i.e. do a 1-step Bellman backup.

        Args:
            V (np.ndarray): list of state values (length = number of states)
            state (Tuple[int, int]): state
            action (Action): action
            gamma (float): discount factor

        Returns:
            ret (float): the expected return
        """
        #print(f"state: {state} action: {action}")
        next_state, reward = self.transitions(state, action)
        #print(f"next state: {next_state} reward: {reward}")
        #print(f"next_state[0]*5+next_state[1]: {next_state[0]*5+next_state[1]}")
        # TODO compute the expected return
        ret = reward + gamma*V[next_state[0]*5+next_state[1]][0]

        return ret

In [4]:
s = (3,3)
u = (0,1)
d = (0,-1)

In [5]:
for x in zip(s,u):
    print(x)
tuple([sum(x) for x in zip(s,d)])

(3, 0)
(3, 1)


(3, 2)

In [6]:
tuple(map(sum, zip(s,d)))

(3, 2)

In [7]:
s[1]
1e-3

0.001

In [30]:
"""
Created a class to contain my policy
"""
class policy:
    "policy model"
    
    def __init__(self, state_space_size, empty: bool = True, random: bool = False) -> None:
        #create a list of lists that starts with all equiprobable actions size of state space
        if not empty and not random:
            self.A = [[0,1,2,3]]*state_space_size
        elif random:
            self.A = [[np.random.randint(0,4)] for x in range(state_space_size)] 
        else:
            self.A = [[]]*state_space_size
        
    def choose_action(self, state) -> Action:
        #return an action with equiprobable chance from available in list for state
        return Action(np.random.choice(self.A[state]))
    
    def action_prob_state(self, state):
        return 1/len(self.A[state])
    
    
"""
Create a function to print Value function in pattern matching the textbook
"""
def print_Value_function(V):
    v = [round(x[0],1) for x in V]
    v = np.array(v)
    v = v.reshape(5,5)
    return np.rot90(v,1)


In [56]:
#iterative policy evaluation

#theta=10-3
#init V(s) randomly for all s
#init V(terminal) to 0
#policy pi_

"""
Loop:
    del = 0
    Loop for each s element of S:
        val_ = V(s)
        V(s) = sum(all a in policy for state s)*sum(dynamics_func*(reward + gamma*V(s_prime)))
        del = max(del, abs(val-V(s)))
    until del < theta
"""

#maybe a class not func
def iter_policy_eval(V, env: Gridworld5x5, pi, gamma=0.9, theta=1e-3):
    converged = False
    t = 0
    
    while not converged:
        print(f"loop: {t}")
        delta = 0
        
        for s in range(len(V)):
            #print(f"s: {s}")
            val = V[s][0]
            #print(f"val: {val}")
            sum_ = 0
            
            for a in Action:
                #print(f'action a: {a}')
                #v = [env.expected_return(V, s, Action(a), gamma), a]
                sum_ += env.expected_return(V, s, Action(a), gamma)
                #print(f"sum per action: {sum_}")
            sum_ = pi.action_prob_state(s)*sum_
            #print(f"pi(a|s): {pi.action_prob_state(s)}")
            
            #print(f"V[s]: {sum_}")
            V[s] = [sum_, pi.choose_action(s)]
            
            #print(f"abs(val-V[s][0]): {abs(val-V[s][0])}")
            delta = max(delta, abs(val-V[s][0]))
            #print(f"delta: {delta}")
            
        if delta < theta:
            #print(f"state: {s}")
            #print(f"converged: {converged} changing to True")
            converged = True
        t+=1
    return V

In [57]:
env = Gridworld5x5()
V = [[0, 0]]*len(env.state_space)
#V[A] = 0
#V[B] = 0
pi = policy(state_space_size=len(env.state_space), empty=False)
A = iter_policy_eval(V,env,pi)

loop: 0
loop: 1
loop: 2
loop: 3
loop: 4
loop: 5
loop: 6
loop: 7
loop: 8
loop: 9
loop: 10
loop: 11
loop: 12
loop: 13
loop: 14
loop: 15
loop: 16
loop: 17
loop: 18


In [58]:
#policy eval Value function
#some of the top right cells have a value that's slightly off
print_Value_function(A)

array([[ 3.3,  8.8,  4.1,  4.5,  1.1],
       [ 1.5,  2.9,  2.1,  1.6,  0.3],
       [ 0. ,  0.7,  0.6,  0.2, -0.5],
       [-1. , -0.5, -0.4, -0.7, -1.3],
       [-1.9, -1.4, -1.3, -1.5, -2. ]])

In [12]:
abs(-4)

4

In [13]:
help(max)

Help on built-in function max in module builtins:

max(...)
    max(iterable, *[, default=obj, key=func]) -> value
    max(arg1, arg2, *args, *[, key=func]) -> value
    
    With a single iterable argument, return its biggest item. The
    default keyword-only argument specifies an object to return if
    the provided iterable is empty.
    With two or more arguments, return the largest argument.



In [14]:
Action.UP

<Action.UP: 3>

In [15]:
mapping = {
    Action.LEFT: (-1, 0),
    Action.DOWN: (0, -1),
    Action.RIGHT: (1, 0),
    Action.UP: (0, 1),
}
mapping[Action.UP]

(0, 1)

In [16]:
actions = [Action.UP, Action.DOWN, Action.RIGHT, Action.LEFT]
np.random.choice(actions)

3

In [17]:
for a in Action:
    print(a)

Action.LEFT
Action.DOWN
Action.RIGHT
Action.UP


In [18]:
V[0][1]

<Action.UP: 3>

In [19]:
env = Gridworld5x5()
env.state_space[env.A[0]*5+env.A[1]]

(1, 4)

In [20]:
env.B

(3, 4)

In [51]:
env.V[env.B[0]*5+env.B[1]]

AttributeError: 'Gridworld5x5' object has no attribute 'V'

In [64]:
#policy iteration

#policy eval (same as above)

#init V arbitrary Real vals
#init pi for all Actions arbitrary
#init V(terminal) = 0

#loop:
#delta = 0
#Loop for each s elmnt of S:
    #val = V[s]
    #V[s] = sig(over s', r)p(s',r|s,pi(s))[r + gamma*V[s']]
    #delta = max(delta, abs(val - V[s]))
#until delta < theta

#policy improvement
"""
policy_stable = True
for each s element of S:
    old_action = pi(s)
    pi(s) = argmax_a Sigma_s',r p(s',r|s,a)[r + gamma*V[s']]
    if old_action != pi(s), then policy_stable = False
if policy_stable, then stop and return V ~ v* and pi ~ pi*; 
else go to policy eval
"""


def policy_iter(V, env: Gridworld5x5, pi, gamma=0.9, theta=1e-1):
    V = iter_policy_eval(V,env,pi,gamma=0.9,theta=1e-3)
    print(f"value function after eval: {V}")

    #Policy Improvement
    policy_stable = True
    for s in range(len(V)):
        print(f"state: {s}")
        old_pi = pi.A
        opt_action = []
        max_val = 0
        for a in Action:
            print(f"action: {a}")
            val = env.expected_return(V,s,a,gamma)
            print(f"val: {val}")
            if max_val == 0:
                if abs(val) >= max_val:
                    max_val = val
                    opt_action = a
            else:
                  if val >= max_val:
                    max_val = val
                    opt_action = a
        pi.A[s] = opt_action
        print(f"optimal action: {pi.A[s]}, old_pi[s]: {old_pi[s]}")
        if old_pi[s] != opt_action:
            print(f"old and new unequal")
            policy_stable = False
    if policy_stable:
        return pi, V
    else:
        policy_iter(V, env, pi)
            
    

In [66]:
env = Gridworld5x5()
V = [[0, 0]]*len(env.state_space)
pi = policy(state_space_size=len(env.state_space), empty=False)
#pi.A
pi, V = policy_iter(V, env, pi)
print_Value_function(V)
#policy iteration is not working entirely, didn't have enough time to fully debug

loop: 0
loop: 1
loop: 2
loop: 3
loop: 4
loop: 5
loop: 6
loop: 7
loop: 8
loop: 9
loop: 10
loop: 11
loop: 12
loop: 13
loop: 14
loop: 15
loop: 16
loop: 17
loop: 18
value function after eval: [[-1.8775782625573882, <Action.LEFT: 0>], [-0.9976846405537394, <Action.UP: 3>], [0.02041919066871256, <Action.DOWN: 1>], [1.488748347580283, <Action.LEFT: 0>], [3.2866696800680235, <Action.LEFT: 0>], [-1.3696981162943427, <Action.LEFT: 0>], [-0.4681527771150198, <Action.LEFT: 0>], [0.690411102617172, <Action.DOWN: 1>], [2.931805720398577, <Action.DOWN: 1>], [8.767271695335092, <Action.UP: 3>], [-1.2619767768354184, <Action.RIGHT: 2>], [-0.4042044758688703, <Action.DOWN: 1>], [0.5840142990588079, <Action.LEFT: 0>], [2.083343030611049, <Action.DOWN: 1>], [4.139819234951403, <Action.LEFT: 0>], [-1.4635940730133385, <Action.LEFT: 0>], [-0.6511903618737772, <Action.RIGHT: 2>], [0.22541640133820975, <Action.LEFT: 0>], [1.6030005626236012, <Action.RIGHT: 2>], [4.5191028547079775, <Action.LEFT: 0>], [-2.0193

array([[ 3.3,  8.8,  4.1,  4.5,  1.1],
       [ 1.5,  2.9,  2.1,  1.6,  0.3],
       [ 0. ,  0.7,  0.6,  0.2, -0.5],
       [-1. , -0.5, -0.4, -0.7, -1.3],
       [-1.9, -1.4, -1.3, -1.5, -2. ]])

In [38]:
#Value Iteration
"""
init:
    theta
    V[s] arbitrary
    V[terminal]=0
    
Loop:
    delta = 0
    Loop for each s element S:
        val = V[s]
        V[s] = max_a sigma_s',r(p(s',r|s,a)[r + gamma*V[s']])
        delta = max(delta, abs(val-V[s]))
until delta < theta
    
output:
deterministic policy, pi~pi*, such that
    pi(s)=argmax_a sigma_s',r(p(s',r|s,a)[r + gamma*V[s']])
"""

def value_iter(V, env: Gridworld5x5, pi, gamma=0.9, theta=1e-3):
    converged = False
    t = 0
    while not converged:
        print(f"loop: {t}")
        delta = 0
        for s in range(len(V)):
            #states 19 and 24
            #print(f"s: {s}")
            val = V[s][0]
            #print(f"val: {val}")
            max_value=[0,0]
            for a in Action:
                #print(f'action a: {a}')
                v = [env.expected_return(V, s, Action(a), gamma), a]
                #if s == 19 or s == 24:
                    #print(f"s: {s} v: {v}")
                if max_value[0] < v[0]:
                    max_value = v
            #if s == 19 or s == 24:
                #print(f"max value: {max_value}")
            V[s] = max_value
            #print(f"abs(val-V[s][0]): {abs(val-V[s][0])}")
            delta = max(delta, abs(val-V[s][0]))
            #print(f"delta: {delta}")
        if delta < theta:
            #print(f"state: {s}")
            #print(f"converged: {converged} changing to True")
            converged = True
        t+=1
    return V

In [43]:
env = Gridworld5x5()
V = [[0, 0]]*len(env.state_space)
#V[A] = 0
#V[B] = 0
pi = policy(state_space_size=len(env.state_space), empty=False)
V = value_iter(V,env,pi)
pi

loop: 0
loop: 1
loop: 2
loop: 3
loop: 4
loop: 5
loop: 6
loop: 7
loop: 8
loop: 9
loop: 10
loop: 11
loop: 12
loop: 13
loop: 14
loop: 15
loop: 16
loop: 17
loop: 18
loop: 19
loop: 20
loop: 21
loop: 22
loop: 23
loop: 24
loop: 25
loop: 26
loop: 27
loop: 28
loop: 29
loop: 30
loop: 31
loop: 32
loop: 33
loop: 34
loop: 35
loop: 36
loop: 37
loop: 38
loop: 39
loop: 40
loop: 41
loop: 42
loop: 43
loop: 44
loop: 45
loop: 46
loop: 47
loop: 48
loop: 49
loop: 50
loop: 51
loop: 52
loop: 53
loop: 54
loop: 55
loop: 56
loop: 57
loop: 58
loop: 59
loop: 60
loop: 61
loop: 62
loop: 63
loop: 64
loop: 65
loop: 66
loop: 67
loop: 68
loop: 69
loop: 70
loop: 71


In [45]:
#all the actions and the values for the output of the value function
#actions are singular, it created a deterministic policy
[[round(v[0], 1), v[1]] for v in V]

[[14.4, <Action.RIGHT: 2>],
 [16.0, <Action.RIGHT: 2>],
 [17.8, <Action.RIGHT: 2>],
 [19.8, <Action.RIGHT: 2>],
 [22.0, <Action.RIGHT: 2>],
 [16.0, <Action.UP: 3>],
 [17.8, <Action.UP: 3>],
 [19.8, <Action.UP: 3>],
 [22.0, <Action.UP: 3>],
 [24.4, <Action.LEFT: 0>],
 [14.4, <Action.LEFT: 0>],
 [16.0, <Action.LEFT: 0>],
 [17.8, <Action.LEFT: 0>],
 [19.8, <Action.LEFT: 0>],
 [22.0, <Action.LEFT: 0>],
 [13.0, <Action.LEFT: 0>],
 [14.4, <Action.LEFT: 0>],
 [16.0, <Action.LEFT: 0>],
 [17.8, <Action.LEFT: 0>],
 [18.0, <Action.LEFT: 0>],
 [11.7, <Action.LEFT: 0>],
 [13.0, <Action.LEFT: 0>],
 [14.4, <Action.LEFT: 0>],
 [16.0, <Action.LEFT: 0>],
 [16.2, <Action.LEFT: 0>]]

In [42]:
#The value iteration function value function output 
#in grid shape matching textbook
#the top row, most right and second most right cells are slightly off
print_Value_function(V)

array([[22. , 24.4, 22. , 18. , 16.2],
       [19.8, 22. , 19.8, 17.8, 16. ],
       [17.8, 19.8, 17.8, 16. , 14.4],
       [16. , 17.8, 16. , 14.4, 13. ],
       [14.4, 16. , 14.4, 13. , 11.7]])

In [48]:
#actions from value iteration arranged in the correct format,
#deterministic policy, singular action
b = [x[1] for x in V]
b = np.array(b)
b = b.reshape(5,5)
np.rot90(b,1)

array([[2, 0, 0, 0, 0],
       [2, 3, 0, 0, 0],
       [2, 3, 0, 0, 0],
       [2, 3, 0, 0, 0],
       [2, 3, 0, 0, 0]])

In [43]:
pi = policy(25, empty=False, random=True)
pi.A

[[3],
 [3],
 [1],
 [0],
 [1],
 [3],
 [1],
 [0],
 [2],
 [0],
 [2],
 [1],
 [1],
 [2],
 [0],
 [2],
 [3],
 [1],
 [2],
 [2],
 [2],
 [0],
 [3],
 [1],
 [2]]

In [44]:
[[np.random.randint(0,4)]]*10
[[np.random.randint(0,4)] for x in range(10)]

[[3], [1], [1], [0], [1], [2], [3], [3], [0], [3]]

In [111]:
pi = policy(25,empty=False, random=True)
pi.A

[[0],
 [3],
 [2],
 [2],
 [2],
 [2],
 [3],
 [2],
 [3],
 [0],
 [0],
 [2],
 [1],
 [0],
 [1],
 [1],
 [2],
 [0],
 [2],
 [1],
 [1],
 [1],
 [1],
 [0],
 [2]]